In [4]:
import random
import math
import pandas as pd
import numpy as np

# Load training data
df_train = pd.read_csv("wifi_training.csv")

# Create input features X and target labels y
X = df_train[["Download", "Upload", "PacketLoss"]]
y = df_train[["Expected"]].rename(columns={"Expected": "O1"})

# Perform Min-Max normalization on input data
X_norm = (X - X.min()) / (X.max() - X.min())

# Check the first rows of the normalized dataset
X_norm.head()

,Download,Upload,PacketLoss
0,0.000000,0.000000,1.000000
1,0.020690,0.038462,0.729412
2,0.034483,0.038462,0.682353
3,0.048276,0.076923,0.552941
4,0.068966,0.076923,0.482353


In [5]:
import random
import math
import pandas as pd
import numpy as np

class NeuralNetwork:
    """
    The neural network does not contain individual neurons or separate layers of neurons.
    There are weights and biases between the layers.
    Weights between layers determine how strongly each neuron of the previous
    layer affects each neuron of the next layer.
    A separate weight indicator is used for each weight.
    Between the layers are weights W1 and W2, which are pandas DataFrames.
    | Weight | Impact                 |
    | ------ | ---------------------- |
    | +0.9   | very strong positive   |
    | +0.2   | weak positive          |
    | 0      | practically no impact  |
    | -0.2   | weak negative          |
    | -0.9   | very strong negative   |
    """
    def __init__(
        self,
        input_size=3,     # Size of the input layer of neurons
        hidden_size=4,    # Size of the hidden layer of neurons
        output_size=1,    # Size of the output layer of neurons
        learning_rate=0.1
    ):
        random.seed(42)
        self.learning_rate = learning_rate

        # Weights between input and hidden layers
        self.W1 = pd.DataFrame(
            [
                [random.gauss(0, 0.1) for _ in range(hidden_size)]
                for _ in range(input_size)
            ],
            index=["Download", "Upload", "PacketLoss"],
            columns=[f"H{i + 1}" for i in range(hidden_size)]
        )

        # Hidden layer bias
        self.b1 = pd.Series(
            [0.0] * hidden_size,
            index=self.W1.columns
        )

        # Weights between hidden and output layers
        self.W2 = pd.DataFrame(
            [
                [random.gauss(0, 0.1) for _ in range(output_size)]
                for _ in range(hidden_size)
            ],
            index=[f"H{i + 1}" for i in range(hidden_size)],
            columns=[f"O{i + 1}" for i in range(output_size)]
        )

        # Output layer bias
        self.b2 = pd.Series(
            [0.0] * output_size,
            index=self.W2.columns
        )

    # Activation function (Sigmoid). Provides a value similar to probability from 0 to 1
    def sigmoid(self, z):
        return z.map(
            lambda value: 1 / (1 + math.exp(-value))
        )

    # Shows how much the neuron weights need to be changed during training
    def sigmoid_derivative(self, value):
        return value * (1 - value)

    # Performs a forward pass of data through the neural network
    def forward(self, X):
        # Input layer → hidden layer
        self.z1 = X.dot(self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        # Hidden layer → output layer
        self.z2 = self.a1.dot(self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    # Training the neural network
    def train(self, X, y, epochs=10000):
        for epoch in range(epochs):
            # Forward propagation
            output = self.forward(X)
            # Error
            error = y - output

            # Output layer gradient
            dz2 = error * self.sigmoid_derivative(output)
            # Weights gradient W2
            dW2 = self.a1.T.dot(dz2)
            # Output layer bias gradient
            db2 = dz2.sum(axis=0)

            # Hidden layer error
            dz1 = pd.DataFrame(
                dz2.to_numpy() @ self.W2.to_numpy().T,
                index=dz2.index,
                columns=self.W1.columns
            ) * self.sigmoid_derivative(self.a1)

            # Weights gradient W1
            dW1 = X.T.dot(dz1)
            # Hidden layer bias gradient
            db1 = dz1.sum(axis=0)

            # Updating weights
            self.W2 += self.learning_rate * dW2
            self.b2 += self.learning_rate * db2
            self.W1 += self.learning_rate * dW1
            self.b1 += self.learning_rate * db1

            # Print error every 500 epochs
            if (epoch + 1) % 500 == 0:
                loss = (error ** 2).mean().mean()
                print(f"Epoch {epoch + 1:5}: loss = {loss:.6f}")

    # Returns the probabilistic result of the neural network's operation
    def predict_proba(self, X):
        return self.forward(X)

    # Gets the final prediction of the neural network: 0 or 1
    def predict(self, X):
        probability = self.predict_proba(X)
        return probability.map(
            lambda value: 1 if value >= 0.5 else 0
        )

In [6]:
# Create the neural network object
nn = NeuralNetwork()

# Start training the neural network on training data for 10000 epochs
nn.train(X_norm, y, epochs=10000)

Epoch   500: loss = 0.032195
Epoch  1000: loss = 0.017846
Epoch  1500: loss = 0.013503
Epoch  2000: loss = 0.011207
Epoch  2500: loss = 0.009664
Epoch  3000: loss = 0.008516
Epoch  3500: loss = 0.007610
Epoch  4000: loss = 0.006868
Epoch  4500: loss = 0.006244
Epoch  5000: loss = 0.005710
Epoch  5500: loss = 0.005247
Epoch  6000: loss = 0.004840
Epoch  6500: loss = 0.004481
Epoch  7000: loss = 0.004160
Epoch  7500: loss = 0.003873
Epoch  8000: loss = 0.003614
Epoch  8500: loss = 0.003379
Epoch  9000: loss = 0.003166
Epoch  9500: loss = 0.002972
Epoch 10000: loss = 0.002795


In [7]:
# Get probabilities and predictions for training data
train_probs = nn.predict_proba(X_norm)
train_preds = nn.predict(X_norm)

# Build the result table
result_train = df_train[["Download", "Upload", "PacketLoss", "Expected"]].copy()
result_train["Probability"] = train_probs.values
result_train["Prediction"] = train_preds.values
result_train["Correct"] = (result_train["Expected"] == result_train["Prediction"])

print("Results on training data:")
result_train.head(10)

Results on training data:


,Download,Upload,PacketLoss,Expected,Probability,Prediction,Correct
0,5,1,8.5,0,1.193532e-08,0,True
1,8,2,6.2,0,1.754437e-08,0,True
2,10,2,5.8,0,2.150706e-08,0,True
3,12,3,4.7,0,3.425458e-08,0,True
4,15,3,4.1,0,5.212921e-08,0,True
5,18,4,3.8,0,1.353915e-07,0,True
6,20,5,3.2,0,3.472042e-07,0,True
7,22,4,2.9,0,3.262723e-07,0,True
8,25,5,2.5,0,1.520939e-06,0,True
9,28,6,2.1,0,1.140996e-05,0,True


In [8]:
# Load new data
df_new = pd.read_csv("wifi_analysis.csv")

# Prepare and normalize new data using training set min/max boundaries
X_new = df_new[["Download", "Upload", "PacketLoss"]]
X_new_norm = (X_new - X.min()) / (X.max() - X.min())

# Make predictions for new data
new_probs = nn.predict_proba(X_new_norm)
new_preds = nn.predict(X_new_norm)

# Build the result table
result_new = df_new[["Download", "Upload", "PacketLoss"]].copy()
result_new["Probability"] = new_probs.values
result_new["Prediction"] = new_preds.values

print("Results on new data (wifi_analysis.csv):")
display(result_new)

Results on new data (wifi_analysis.csv):


,Download,Upload,PacketLoss,Probability,Prediction
0,7,1,7.8,1.388052e-08,0
1,11,2,5.5,2.384685e-08,0
2,17,3,4.5,7.695586e-08,0
3,21,4,3.6,2.727186e-07,0
4,27,5,2.8,3.445218e-06,0
5,33,6,2.0,1.546912e-04,0
6,39,7,1.4,2.695239e-02,0
7,42,6,2.5,7.466599e-02,0
8,47,7,1.9,9.250704e-01,1
9,52,8,1.6,9.991317e-01,1
